# Convulational neural networks with PyTorch

## PyTorch Setup

In [96]:
import torch

In [97]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

Using device: cuda


## Load the dataset

In [98]:
from torchvision.datasets import CIFAR10
from torchvision import transforms

In [99]:
transform = transforms.ToTensor()
train_data = CIFAR10(root="./data", train=True, download=True, transform=transform)
test_data = CIFAR10(root="./data", train=False, download=True, transform=transform)

c:\Users\oscar\repos\clases-cetys\computational-intelligence\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [100]:
train_data

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [101]:
test_data

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: ToTensor()

## Defining the neural network

In [102]:
import torch.nn as nn
import torch.optim as optim

In [103]:
input_shape = train_data[0][0].shape
input_shape


torch.Size([3, 32, 32])

In [104]:
output_shape = train_data.classes.__len__()
print(output_shape)

10


In [113]:
model1 = nn.Sequential(
    nn.Conv2d(input_shape[0], 16, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),
    nn.Linear(64 * 4 * 4, output_shape)
).to(device)

model2 = nn.Sequential(
    nn.Conv2d(input_shape[0], 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),
    nn.Dropout(0.30),
    nn.Linear(128 * 4 * 4, output_shape)
).to(device)

model3 = nn.Sequential(
    nn.Conv2d(input_shape[0], 24, kernel_size=5, stride=1, padding=2),
    nn.ReLU(),
    nn.AvgPool2d(kernel_size=2, stride=2),

    nn.Conv2d(24, 48, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.AvgPool2d(kernel_size=2, stride=2),

    nn.Conv2d(48, 96, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),
    nn.Linear(96 * 4 * 4, 256),
    nn.ReLU(),
    nn.Dropout(0.40),
    nn.Linear(256, output_shape)
).to(device)

optimizer2 = optim.Adam(model2.parameters(), lr=1e-3)
optimizer3 = optim.Adam(model3.parameters(), lr=1e-3)


loss_function = nn.CrossEntropyLoss()
optimizer1 = optim.Adam(model1.parameters())

## Mini-batches function

In [106]:
import numpy as np

In [107]:
def create_minibatches(batch_size, x, y):
    total_data = len(x)
    indices = np.arange(total_data)

    for i in range(0, total_data, batch_size):
        batch_idx = indices[i:i + batch_size]
        x_batch = torch.stack([x[j] for j in batch_idx])
        y_batch = torch.tensor([y[j] for j in batch_idx], dtype=torch.long)
        yield x_batch, y_batch

## Train function

In [108]:
from tqdm import tqdm

In [109]:
def train(model, train_data, val_data, optimizer, criterion, epochs, batch_size, patience=None, delta=0.005):
    # Training data
    x_data = [x for x, _ in train_data]
    y_data = [y for _, y in train_data]

    # Validation data
    x_val = [x for x, y in val_data]
    y_val = [y for _, y in val_data]

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        epoch_iterator = tqdm(
            create_minibatches(batch_size, x_data, y_data),
            total=(len(train_data) + batch_size - 1) // batch_size,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for inputs, labels in epoch_iterator:
            optimizer.zero_grad()
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Accuracy
            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

            running_loss += loss.item() * len(inputs)

        epoch_train_loss = running_loss / len(train_data)
        epoch_train_accuracy = train_correct / train_total

        # Eval
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in create_minibatches(batch_size, x_val, y_val):
                if inputs.device != device:
                    inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                # Accuracy
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

                val_loss += loss.item() * len(inputs)
        
        epoch_val_loss = val_loss / len(val_data)
        epoch_val_accuracy = val_correct / val_total

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Train Acc: {epoch_train_accuracy:.4f}, Val Acc: {epoch_val_accuracy:.4f}")

        # Early stopping
        if patience is not None:
            if epoch == 0:
                best_val_loss = epoch_val_loss
                patience_counter = 0
            else:
                if epoch_val_loss < best_val_loss - delta:
                    best_val_loss = epoch_val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stop triggered")
                break

    return epoch_train_loss, epoch_val_loss, epoch_train_accuracy, epoch_val_accuracy

In [110]:
model1_metrics = train(
    model=model1,
    train_data=train_data,
    val_data=test_data,
    optimizer=optimizer1,
    criterion=loss_function,
    epochs=30,
    batch_size=64,
    patience=5
)
model1_metrics

Epoch 1/30: 100%|██████████| 782/782 [00:03<00:00, 201.39it/s]


Epoch [1/30], Loss: 1.6372, Val Loss: 1.4088, Train Acc: 0.4109, Val Acc: 0.4918


Epoch 2/30: 100%|██████████| 782/782 [00:03<00:00, 217.62it/s]


Epoch [2/30], Loss: 1.3007, Val Loss: 1.2238, Train Acc: 0.5398, Val Acc: 0.5609


Epoch 3/30: 100%|██████████| 782/782 [00:03<00:00, 212.90it/s]


...

Epoch 18/30: 100%|██████████| 782/782 [00:02<00:00, 292.10it/s]


Epoch [18/30], Loss: 0.6057, Val Loss: 0.9004, Train Acc: 0.7929, Val Acc: 0.7116


Epoch 19/30: 100%|██████████| 782/782 [00:02<00:00, 292.95it/s]


Epoch [19/30], Loss: 0.5931, Val Loss: 0.9017, Train Acc: 0.7968, Val Acc: 0.7107
Early stop triggered


(0.5931368998432159, 0.9017264465332031, 0.79676, 0.7107)

In [114]:
model2_metrics = train(
    model=model2,
    train_data=train_data,
    val_data=test_data,
    optimizer=optimizer2,
    criterion=loss_function,
    epochs=30,
    batch_size=64,
    patience=5
)
model2_metrics

Epoch 1/30: 100%|██████████| 782/782 [00:05<00:00, 134.40it/s]


Epoch [1/30], Loss: 1.5746, Val Loss: 1.2795, Train Acc: 0.4321, Val Acc: 0.5435


Epoch 2/30: 100%|██████████| 782/782 [00:05<00:00, 142.14it/s]


Epoch [2/30], Loss: 1.2198, Val Loss: 1.0616, Train Acc: 0.5691, Val Acc: 0.6281


Epoch 3/30: 100%|██████████| 782/782 [00:05<00:00, 144.35it/s]


...

Epoch 21/30: 100%|██████████| 782/782 [00:03<00:00, 259.58it/s]


Epoch [21/30], Loss: 0.5302, Val Loss: 0.7520, Train Acc: 0.8110, Val Acc: 0.7534


Epoch 22/30: 100%|██████████| 782/782 [00:03<00:00, 245.57it/s]


Epoch [22/30], Loss: 0.5159, Val Loss: 0.7381, Train Acc: 0.8173, Val Acc: 0.7593
Early stop triggered


(0.5158717366981507, 0.7381362643241882, 0.8173, 0.7593)

In [115]:
model3_metrics = train(
    model=model3,
    train_data=train_data,
    val_data=test_data,
    optimizer=optimizer3,
    criterion=loss_function,
    epochs=30,
    batch_size=64,
    patience=5
)

Epoch 1/30: 100%|██████████| 782/782 [00:06<00:00, 112.01it/s]


Epoch [1/30], Loss: 1.6665, Val Loss: 1.3633, Train Acc: 0.3895, Val Acc: 0.5010


Epoch 2/30: 100%|██████████| 782/782 [00:06<00:00, 117.36it/s]


Epoch [2/30], Loss: 1.3504, Val Loss: 1.2082, Train Acc: 0.5134, Val Acc: 0.5649


Epoch 3/30: 100%|██████████| 782/782 [00:06<00:00, 115.60it/s]


...

Epoch [13/30], Loss: 0.6656, Val Loss: 0.9661, Train Acc: 0.7619, Val Acc: 0.6853


Epoch 14/30: 100%|██████████| 782/782 [00:03<00:00, 228.28it/s]


Epoch [14/30], Loss: 0.6318, Val Loss: 0.9916, Train Acc: 0.7729, Val Acc: 0.6799


Epoch 15/30: 100%|██████████| 782/782 [00:03<00:00, 229.91it/s]


Epoch [15/30], Loss: 0.6082, Val Loss: 0.9894, Train Acc: 0.7799, Val Acc: 0.6953
Early stop triggered


## Models evaluation

In [130]:
def evaluate_model(model, dataset, criterion, batch_size=256):
    x_data = [x for x, _ in dataset]
    y_data = [y for _, y in dataset]

    model.eval()
    total_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in create_minibatches(batch_size, x_data, y_data):
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    return {
        "test_loss": total_loss / test_total,
        "test_accuracy": test_correct / test_total,
    }


models = {
    "Model 1": model1,
    "Model 2": model2,
    "Model 3": model3,
}

results = {}
for name, model in models.items():
    metrics = evaluate_model(model, test_data, loss_function, batch_size=256)
    results[name] = metrics
    print(
        f"{name} = Test Loss: {metrics['test_loss']:.4f}, Test Accuracy: {metrics['test_accuracy']:.4f}"
    )

# Get the best model
best_name = max(results, key=lambda k: results[k]["test_accuracy"])
best_metrics = results[best_name]

print("\nBest model:", best_name)
print(
    f"{best_name} Test Loss: {best_metrics['test_loss']:.4f}, {best_name} Test Accuracy: {best_metrics['test_accuracy']:.4f}"
)

Model 1 = Test Loss: 2.3030, Test Accuracy: 0.1058
Model 2 = Test Loss: 0.7381, Test Accuracy: 0.7593
Model 3 = Test Loss: 0.9894, Test Accuracy: 0.6953

Best model: Model 2
Model 2 Test Loss: 0.7381, Model 2 Test Accuracy: 0.7593


## Reflection

In this practice, I learned how convolutional neural networks can detect specific image patterns even when those patterns appear in different positions. I had problems understanding convolutional layers and tracking input/output dimensions across the network but it eventualy become clearer.

After comparing the three architectures, Model 2 performed the best overall. It achieved better accuracy with lower loss.